In [ ]:
# ═══════════════════════════════════════════════════════════════════
# BorsaBot — 01 | Veri İndirme (Binance Public REST API)
# Colab'da çalıştır. Auth gerekmez, tamamen ücretsiz.
# ═══════════════════════════════════════════════════════════════════

In [ ]:
# ─────────────────────────────────────────────────────────────────
!pip install -q requests pandas numpy pyarrow tqdm

import os, time, math, itertools, pickle, warnings
from pathlib import Path
import requests
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
print("✓ Paketler hazır")

In [ ]:
# ─────────────────────────────────────────────────────────────────
CFG = dict(
    # Semboller ve zaman dilimi
    symbols    = ["BTCUSDT", "ETHUSDT"],
    interval   = "1h",          # 1m 5m 15m 1h 4h 1d
    start_date = "2021-01-01",
    end_date   = "2024-12-31",

    # Saklama
    raw_dir    = "/content/borsabot_data/raw",
    out_dir    = "/content/borsabot_data/processed",
    model_dir  = "/content/borsabot_data/models",
)

for d in [CFG["raw_dir"], CFG["out_dir"], CFG["model_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✓ Dizinler oluşturuldu")
print(f"  Semboller : {CFG['symbols']}")
print(f"  Aralık    : {CFG['start_date']} → {CFG['end_date']}")
print(f"  Zaman dil : {CFG['interval']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
BINANCE_BASE = "https://api.binance.com/api/v3/klines"
COLS = ["open_time","open","high","low","close","volume",
        "close_time","qav","n_trades","taker_buy_base",
        "taker_buy_quote","ignore"]

def _ts(dt_str: str) -> int:
    return int(pd.Timestamp(dt_str).timestamp() * 1000)

def fetch_klines(symbol: str, interval: str,
                 start_ms: int, end_ms: int,
                 limit: int = 1000) -> pd.DataFrame:
    """Binance'ten tek batch OHLCV çeker."""
    resp = requests.get(BINANCE_BASE, params=dict(
        symbol=symbol, interval=interval,
        startTime=start_ms, endTime=end_ms,
        limit=limit,
    ), timeout=15)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data, columns=COLS)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    for c in ["open","high","low","close","volume"]:
        df[c] = df[c].astype(float)
    return df.set_index("open_time")[["open","high","low","close","volume"]]

def download_symbol(symbol: str, interval: str,
                    start_date: str, end_date: str,
                    raw_dir: str) -> pd.DataFrame:
    """Tüm tarih aralığını loop ile çeker ve birleştirir."""
    path = Path(raw_dir) / f"{symbol}_{interval}.parquet"
    if path.exists():
        print(f"  [{symbol}] Cache'den yükleniyor ...")
        return pd.read_parquet(path)

    start_ms = _ts(start_date)
    end_ms   = _ts(end_date)
    frames   = []

    # Interval → millisaniye
    _mult = {"m":60,"h":3600,"d":86400,"w":604800}
    num   = int(interval[:-1])
    unit  = _mult[interval[-1]]
    step  = num * unit * 1000 * 1000  # 1000 bar

    cursor = start_ms
    pbar   = tqdm(desc=f"  [{symbol}] İndiriliyor", unit="batch")
    while cursor < end_ms:
        batch = fetch_klines(symbol, interval, cursor,
                             min(cursor + step - 1, end_ms))
        if batch.empty:
            break
        frames.append(batch)
        cursor = int(batch.index[-1].timestamp() * 1000) + 1
        pbar.update(1)
        time.sleep(0.15)   # rate-limit koruması
    pbar.close()

    df = pd.concat(frames).sort_index()
    df = df[~df.index.duplicated(keep="first")]
    df.to_parquet(path)
    print(f"  [{symbol}] {len(df):,} bar → {path.name}")
    return df

In [ ]:
# ─────────────────────────────────────────────────────────────────
ohlcv: dict[str, pd.DataFrame] = {}

for sym in CFG["symbols"]:
    ohlcv[sym] = download_symbol(
        sym, CFG["interval"],
        CFG["start_date"], CFG["end_date"],
        CFG["raw_dir"],
    )

# Özet
for sym, df in ohlcv.items():
    print(f"\n{sym}: {len(df):,} bar  |  "
          f"{df.index[0].date()} → {df.index[-1].date()}")
    print(df.tail(3))